In [67]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVR
from sklearn.ensemble import RandomForestRegressor

In [68]:
data = pd.read_csv("AmesHousing.csv")

In [69]:
#Data preparation set (one hot encoded)
data["MS SubClass"] = data["MS SubClass"].astype('category')
data["MS Zoning"] = data["MS Zoning"].astype('category')
cat_columns = data.select_dtypes(['category']).columns
data[cat_columns] = data[cat_columns].apply(lambda x: x.cat.codes)

In [70]:
#Feature selection
subset = data[["MS SubClass","MS Zoning","Lot Frontage", "Lot Area", "Overall Qual", "Overall Cond", "Year Built", "Year Remod/Add","SalePrice"]]
subset = subset.dropna()
display(subset)

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,SalePrice
0,0,5,141.0,31770,6,5,1960,1960,215000
1,0,4,80.0,11622,5,6,1961,1961,105000
2,0,5,81.0,14267,6,6,1958,1958,172000
3,0,5,93.0,11160,7,5,1968,1968,244000
4,5,5,74.0,13830,5,5,1997,1998,189900
...,...,...,...,...,...,...,...,...,...
2924,0,5,160.0,20000,5,7,1960,1996,131000
2925,8,5,37.0,7937,6,6,1984,1984,142500
2927,9,5,62.0,10441,5,5,1992,1992,132000
2928,0,5,77.0,10010,5,5,1974,1975,170000


In [71]:
#Set ground truth labels
Y = pd.DataFrame(subset["SalePrice"])
display(Y)

,SalePrice
0,215000
1,105000
2,172000
3,244000
4,189900
...,...
2924,131000
2925,142500
2927,132000
2928,170000


In [72]:
#Remove removes rows with missing values
subset.drop(inplace=True,columns=["SalePrice"])
X = subset
display(X)

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add
0,0,5,141.0,31770,6,5,1960,1960
1,0,4,80.0,11622,5,6,1961,1961
2,0,5,81.0,14267,6,6,1958,1958
3,0,5,93.0,11160,7,5,1968,1968
4,5,5,74.0,13830,5,5,1997,1998
...,...,...,...,...,...,...,...,...
2924,0,5,160.0,20000,5,7,1960,1996
2925,8,5,37.0,7937,6,6,1984,1984
2927,9,5,62.0,10441,5,5,1992,1992
2928,0,5,77.0,10010,5,5,1974,1975


In [73]:
#train test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.20, random_state=42)

In [74]:
#Save ground truths to CSV file
Y_test = pd.DataFrame(Y_test)
Y_test.to_csv("truth.csv")
display(Y_test)

,SalePrice
1922,150000
1409,235000
2597,108480
1784,311872
1374,134432
...,...
1044,89500
1980,130000
277,64000
520,250000


## Support Vector Regressor

In [75]:
#Create SVR model
lsvr = LinearSVR(random_state=0, tol=1e-05)

In [76]:
#Train model
lsvr.fit(X_train,Y_train.values.ravel())

/Users/dustin/anaconda3/lib/python3.11/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/Users/dustin/anaconda3/lib/python3.11/site-packages/sklearn/svm/_base.py:1242: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVR(random_state=0, tol=1e-05)

In [77]:
#Display feature importance
display(lsvr.coef_)

array([ 690.96912534, -273.1490447 ,  403.85500689,    1.53971619,
       1140.3506115 ,  -60.32778662,  566.56952929, -525.85016136])

In [78]:
#Do the predictions
l_pred = lsvr.predict(X_test)
l_pred= pd.DataFrame(l_pred)
l_pred.rename(columns={0:"prediction"},inplace=True)
l_pred.to_csv("SVR_prediction.csv")
display(l_pred)

,prediction
0,139598.778514
1,130764.837539
2,90709.241712
3,134432.870184
4,144147.321536
...,...
483,105321.892616
484,113317.050202
485,138018.467564
486,155246.078263


## Random Forest Regressor

In [79]:
#Create teh RF model
reg = RandomForestRegressor(max_depth=2,random_state=0)

In [80]:
#Train the model
reg.fit(X_train,Y_train.values.ravel())

RandomForestRegressor(max_depth=2, random_state=0)

In [81]:
#Do teh predicitons
r_pred= reg.predict(X_test)
r_pred= pd.DataFrame(r_pred)
r_pred.rename(columns={0:"prediction"},inplace=True)
r_pred.to_csv("RFR_prediction.csv")
display(r_pred)

,prediction
0,139446.229530
1,139446.229530
2,135587.686156
3,199841.458501
4,135587.686156
...,...
483,139446.229530
484,135587.686156
485,135587.686156
486,389489.203700


In [82]:
import sklearn.metrics as met
#Calculate MAE metric
display(met.mean_absolute_error(y_true=Y_test,y_pred=l_pred))
display(met.mean_absolute_error(y_true=Y_test,y_pred=r_pred))

64072.413477795686

32409.35493503281

In [83]:
#Calculate MAE metric
display(met.r2_score(y_true=Y_test,y_pred=l_pred))
display(met.r2_score(y_true=Y_test,y_pred=r_pred))

-0.25751659374530944

0.700098454144707